# Definning the development data

## Importing packages

In [1]:
# Data handling and manipulation
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("Data/pd_modelling.csv")

In [3]:
df0 = df.copy()

In [4]:
dictionary = pd.read_csv("Data/data_dictionary.csv")

## Sampling the risk differentiation data

In [5]:
# Extracting the quarter number from the quarter string
df["quarter_number"] = df["quarter"].apply(lambda x: int(x.split("Q")[1]))

# Setting random stage for reproducibility
rng = np.random.default_rng(42)

# Function to randomly select a quarter for each customer
def quarter_roll(series):
    unique_values = np.unique(series)
    return rng.choice(unique_values)

# Assigning a random quarter to each customer
df["random_quarter"] = df.groupby("customer_id")["quarter_number"].transform(quarter_roll)

# Filtering the dataframe to include only the rows where the quarter number matches the randomly assigned quarter
riskdiff_data = df[df["quarter_number"] == df["random_quarter"]].copy()

C:\Users\niels\AppData\Local\Temp\ipykernel_20148\2913179472.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["quarter_number"] = df["quarter"].apply(lambda x: int(x.split("Q")[1]))
C:\Users\niels\AppData\Local\Temp\ipykernel_20148\2913179472.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["random_quarter"] = df.groupby("customer_id")["quarter_number"].transform(quarter_roll)


## Splitting the risk differentation data

In [6]:
riskdiff_data["port_type"] = np.where(riskdiff_data["quarter"] > '2019Q4', "OOT", "DEV")
test_indices = train_test_split(riskdiff_data[riskdiff_data["port_type"] == "DEV"], test_size=0.2, random_state=42, shuffle=True)[1].index
riskdiff_data.loc[test_indices, "port_type"] = "OOS"

## Saving the risk differentation data

In [7]:
riskdiff_data.to_csv("Data/riskdiff_data.csv", index=False)